# NoPOS — HPC Setup & Job Submission Guide

This notebook walks you through transferring the project to your university HPC cluster (SLURM / `sbatch`) and running training on an H100 GPU.

**Outline:**
1. Connect to the login node
2. Set up a Conda environment
3. Clone the repository & install dependencies
4. Download and preprocess WikiText-103
5. Write an `sbatch` job script
6. Submit, monitor, and cancel jobs
7. Retrieve results

> **Note:** All shell commands below are meant to be run **on the HPC login node** (after SSH-ing in), not on your local machine — unless explicitly labelled *"(local)"*.

## Step 1 — Connect to the Login Node

Open a terminal on your **local machine** and SSH in:

```bash
# (local) Replace <username> and <hpc.university.edu> with your credentials
ssh <username>@<hpc.university.edu>
```

If your cluster uses a jump host or VPN, connect to that first. Once logged in you will land on the **login node** — a shared machine used only for setup and job submission, **not** for heavy computation.

## Step 2 — Set Up a Conda Environment

Most HPC clusters provide Conda/Miniconda via the module system. Load it and create a dedicated environment for this project.

```bash
# Load the conda module (exact name varies by cluster — check with: module avail conda)
module load anaconda3      # or: module load miniconda3

# Create a Python 3.10 environment (only needed once)
conda create -n nopos python=3.10 -y

# Activate it (do this every time you log in or start a new job)
conda activate nopos
```

> **Tip:** Add `conda activate nopos` to your `~/.bashrc` so it activates automatically on login.

## Step 3 — Clone the Repository & Install Dependencies

### 3a. Clone the repo

```bash
# Pick a suitable directory on your cluster's scratch/work filesystem
# (avoid your home directory for large files — check your cluster docs)
cd $SCRATCH    # e.g. /scratch/<username>  or  /work/<username>

git clone --branch main https://github.com/Anxinal/futureMask.git nopos
cd nopos
```

### 3b. Install PyTorch (CUDA build)

Check the CUDA version available on the compute nodes first:

```bash
# Run on the login node or request an interactive session to check
nvidia-smi   # shows driver version → infer CUDA version
```

Then install PyTorch matching that CUDA version:

```bash
# Example for CUDA 12.x (H100 typically ships with CUDA 12+)
pip install "torch>=2.7.0" --index-url https://download.pytorch.org/whl/cu121
```

### 3c. Install NumPy

```bash
pip install numpy
```

### 3d. Install fairseq (editable, bypassing pyproject.toml)

This mirrors the exact workaround used in `colab.ipynb`:

```bash
cd $SCRATCH/nopos
mv pyproject.toml pyproject.toml.bak
pip install -e . --no-build-isolation
mv pyproject.toml.bak pyproject.toml

# Verify
python -c "import fairseq; print('fairseq OK:', fairseq.__version__)"
```

## Step 4 — Download & Preprocess WikiText-103

This only needs to be done **once**. The preprocessed binary data is then reused by all job submissions.

```bash
cd $SCRATCH/nopos

# Install the HuggingFace datasets helper
pip install -q datasets

# Download raw text
mkdir -p /scratch/$USER/wt103-raw/wikitext-103
python scripts/download_wikitext.py   # writes wiki.{train,valid,test}.tokens

# Binarise with fairseq (produces data-bin/)
python -m fairseq_cli.preprocess \
    --only-source \
    --trainpref /scratch/$USER/wt103-raw/wikitext-103/wiki.train.tokens \
    --validpref /scratch/$USER/wt103-raw/wikitext-103/wiki.valid.tokens \
    --testpref  /scratch/$USER/wt103-raw/wikitext-103/wiki.test.tokens \
    --destdir   $SCRATCH/nopos/data-bin/wikitext-103 \
    --workers 4
```

After this step you should see `train.bin`, `train.idx`, `valid.bin`, etc. inside `data-bin/wikitext-103/`.

> **Alternatively**, if you already have the preprocessed data locally (e.g. from Colab/Drive), copy it with `scp` or `rsync`:
> ```bash
> # (local) upload your local data-bin to the cluster
> rsync -avz --progress ./data-bin/wikitext-103/ \
>     <username>@<hpc.university.edu>:$SCRATCH/nopos/data-bin/wikitext-103/
> ```